# Train NotAFigurine Classifier (One-Class Autoencoder)

Train a **one-class autoencoder** on writing elements to detect anomalies (non-writing elements).

**How it works:**
- Learns to reconstruct HOG features of writing elements (letters, digits, punctuation)
- Low reconstruction error → writing element (positive)
- High reconstruction error → not a writing element (negative/anomaly)

**Input:** `writing_elements_classifier.zip` (all classes: lowercase, uppercase, digit, punctuation)

**Output:** `notafigurine_classifier.tflite` — autoencoder model for inference

## Step 1 — Upload writing_elements zip

In [ ]:
import os
from google.colab import files

print('Upload writing_elements_classifier.zip:')
uploaded = files.upload()

WRITING_ZIP_PATH = None
for filename in uploaded.keys():
    if 'writing' in filename.lower():
        WRITING_ZIP_PATH = filename

if not WRITING_ZIP_PATH:
    raise FileNotFoundError('writing_elements_classifier.zip not found')

print(f'✅ Loaded: {WRITING_ZIP_PATH}')

## Step 2 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy scikit-image scikit-learn matplotlib

## Step 3 — Extract and index glyphs

In [ ]:
import zipfile
import shutil
import numpy as np
from PIL import Image

# Extract zip
EXTRACT_DIR = '/tmp/writing_elements_extracted'
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)

with zipfile.ZipFile(WRITING_ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Find glyphs folder
GLYPHS_DIR = None
for root, dirs, files_list in os.walk(EXTRACT_DIR):
    if 'glyphs' in dirs:
        GLYPHS_DIR = os.path.join(root, 'glyphs')
        break

if not GLYPHS_DIR:
    raise FileNotFoundError('glyphs/ folder not found')

print(f'✅ Found glyphs: {GLYPHS_DIR}')

# Count and index glyphs
all_glyph_paths = []
class_counts = {}

for class_dir in os.listdir(GLYPHS_DIR):
    class_path = os.path.join(GLYPHS_DIR, class_dir)
    if not os.path.isdir(class_path):
        continue
    
    class_glyphs = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.png')]
    all_glyph_paths.extend(class_glyphs)
    class_counts[class_dir] = len(class_glyphs)

print(f'\nWriting element glyphs:')
for cls, count in sorted(class_counts.items()):
    print(f'  {cls}: {count}')
print(f'  Total: {len(all_glyph_paths)}')

## Step 4 — HOG feature extraction

In [ ]:
import gc
import ctypes

IMG_SIZE = 32
_ORIENTATIONS = 9
_PX_PER_CELL = 4
_CPB = 2
_N_CELLS = IMG_SIZE // _PX_PER_CELL
_N_BLOCKS = _N_CELLS - _CPB + 1
_BLOCK_SIZE = _CPB * _CPB * _ORIENTATIONS
FEATURE_DIM = _N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE + 3

_CY = np.repeat(np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_CX = np.tile(np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_BASE = (_CY * _N_CELLS + _CX) * _ORIENTATIONS

try:
    _libc = ctypes.CDLL('libc.so.6')
    _has_malloc_trim = True
except:
    _has_malloc_trim = False

def _malloc_trim():
    gc.collect()
    if _has_malloc_trim:
        _libc.malloc_trim(0)

def _compute_hog(img_32x32):
    gx = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gy = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gx[:, 1:-1] = img_32x32[:, 2:] - img_32x32[:, :-2]
    gy[1:-1, :] = img_32x32[2:, :] - img_32x32[:-2, :]

    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0

    bin_width = 180.0 / _ORIENTATIONS
    bf = ang.ravel() / bin_width
    b0 = bf.astype(np.int32) % _ORIENTATIONS
    b1 = (b0 + 1) % _ORIENTATIONS
    t = bf - b0
    mf = mag.ravel()

    flat = np.bincount(
        np.concatenate([_BASE + b0, _BASE + b1]),
        weights=np.concatenate([mf * (1.0 - t), mf * t]),
        minlength=_N_CELLS * _N_CELLS * _ORIENTATIONS,
    )
    cell_hists = flat.reshape(_N_CELLS, _N_CELLS, _ORIENTATIONS)

    eps2 = 1e-5 ** 2
    hog_out = np.empty(_N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE, dtype=np.float64)
    out_i = 0
    for by in range(_N_BLOCKS):
        for bx in range(_N_BLOCKS):
            block = cell_hists[by:by + _CPB, bx:bx + _CPB, :].ravel().copy()
            block /= np.sqrt(np.dot(block, block) + eps2)
            np.clip(block, 0.0, 0.2, out=block)
            block /= np.sqrt(np.dot(block, block) + eps2)
            hog_out[out_i:out_i + _BLOCK_SIZE] = block
            out_i += _BLOCK_SIZE
    return hog_out

def extract_features(arr_32x32, orig_w, orig_h):
    img_gray = arr_32x32.astype(np.float64)
    hog_feat = _compute_hog(img_gray)
    extra = np.array([orig_w / max(orig_h, 1), np.mean(img_gray), np.std(img_gray)], dtype=np.float64)
    return np.concatenate([hog_feat, extra]).astype(np.float32)

print(f'✅ HOG extractor ready (dim={FEATURE_DIM})')

## Step 5 — Load and extract HOG features

In [ ]:
print('Loading glyphs and extracting HOG features...')

X_all = []
loaded_count = 0
error_count = 0

for i, glyph_path in enumerate(all_glyph_paths):
    try:
        with Image.open(glyph_path) as img:
            orig_w, orig_h = img.width, img.height
            arr = np.array(img.convert('L').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
        features = extract_features(arr, orig_w, orig_h)
        X_all.append(features)
        loaded_count += 1
    except Exception as e:
        error_count += 1
    
    if (i + 1) % max(1, len(all_glyph_paths) // 10) == 0:
        pct = 100 * (i + 1) // len(all_glyph_paths)
        print(f'  {pct}% ({loaded_count} loaded)', flush=True)

X = np.array(X_all, dtype=np.float32)

print(f'\n✅ Loaded {loaded_count} glyphs ({error_count} errors)')
print(f'   Shape: {X.shape}  ({X.nbytes / 1e9:.2f} GB)')

_malloc_trim()

## Step 6 — Normalize features and train autoencoder

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Normalize
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std == 0] = 1.0
X_normalized = (X - X_mean) / X_std

# Split
X_train, X_val = train_test_split(X_normalized, test_size=0.15, random_state=42)

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

# Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(FEATURE_DIM),
], name='notafigurine_autoencoder')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
model.summary()

history = model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=100,
    batch_size=128,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6, monitor='val_loss'),
    ],
    verbose=1,
)

print(f'\n✅ Best val loss: {min(history.history["val_loss"]):.6f}')

## Step 7 — Analyze reconstruction errors

In [ ]:
import matplotlib.pyplot as plt

X_reconstructed = model.predict(X_normalized, verbose=0)
reconstruction_errors = np.mean((X_normalized - X_reconstructed) ** 2, axis=1)

print(f'Reconstruction error stats:')
print(f'  Mean: {reconstruction_errors.mean():.6f}')
print(f'  Std: {reconstruction_errors.std():.6f}')
print(f'  Min: {reconstruction_errors.min():.6f}')
print(f'  Max: {reconstruction_errors.max():.6f}')
print(f'  95th percentile: {np.percentile(reconstruction_errors, 95):.6f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(reconstruction_errors, bins=50, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Reconstruction Error')
ax1.set_ylabel('Frequency')
ax1.set_title('Error Distribution')
ax1.axvline(np.percentile(reconstruction_errors, 95), color='r', linestyle='--', label='95th %ile')
ax1.legend()

ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Training')
ax2.legend()
ax2.set_yscale('log')
plt.tight_layout()
plt.show()

## Step 8 — Export TFLite

In [ ]:
TFLITE_PATH = 'notafigurine_classifier.tflite'

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
print(f'  Input : {interp.get_input_details()[0]["shape"]}')
print(f'  Output: {interp.get_output_details()[0]["shape"]}')

## Step 9 — Download

In [ ]:
files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')